# Downloading KEGG Signalling Pathway Annotations

Created by Ines on the 4th November, 2025.

In [ ]:
#!/usr/bin/env python

"""
Download gene-pathway mappings from KEGG REST API
Generates a CSV file suitable for pathway_distance.py
"""

import requests
import pandas as pd
import time
import argparse
import sys
import os
from typing import Dict, Set, List

## Original Script - Slow downloader which overseeds rate limit when downloading all kegg pathways PLUS translating to symbols

In [ ]:


# KEGG organism codes: https://www.genome.jp/kegg/catalog/org_list.html
KEGG_ORGANISM_CODES = {
    'human': 'hsa',
    'mouse': 'mmu',
    'rat': 'rno',
    'fruit_fly': 'dme',
    'worm': 'cel',
    'yeast': 'sce',
    'zebrafish': 'dre',
}

class KEGGDownloader:
    """Download pathway and gene data from KEGG REST API"""
    
    def __init__(self, organism_code: str = 'hsa', rate_limit_delay: float = 0.4):
        """
        Initialize KEGG downloader
        
        Args:
            organism_code: 3-letter KEGG organism code (default: hsa for human)
            rate_limit_delay: Delay between API calls in seconds (KEGG requests <3 calls/sec)
        """
        self.organism_code = organism_code
        self.rate_limit_delay = rate_limit_delay
        self.base_url = 'https://rest.kegg.jp'
        self.last_request_time = 0
    
    def _rate_limited_request(self, url: str) -> str:
        """Make a rate-limited request to KEGG API"""
        # Enforce rate limit (KEGG allows max 3 calls per second)
        elapsed = time.time() - self.last_request_time
        if elapsed < self.rate_limit_delay:
            time.sleep(self.rate_limit_delay - elapsed)
        
        self.last_request_time = time.time()
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"ERROR fetching {url}: {str(e)}")
            return None
    
    def get_all_pathways(self) -> Dict[str, str]:
        """
        Get all pathways for the organism
        Returns: {pathway_id: pathway_name}
        """
        print(f"Fetching pathway list for {self.organism_code}...")
        
        url = f"{self.base_url}/list/pathway/{self.organism_code}"
        response = self._rate_limited_request(url)
        
        if not response:
            return {}
        
        pathways = {}
        for line in response.strip().split('\n'):
            if line:
                parts = line.split('\t')
                if len(parts) >= 2:
                    pathway_id = parts[0]  # e.g., "path:hsa05200"
                    pathway_name = parts[1]  # e.g., "Pathways in cancer"
                    pathways[pathway_id] = pathway_name
        
        print(f"Found {len(pathways)} pathways")
        return pathways
    
    def get_genes_in_pathway(self, pathway_id: str) -> Set[str]:
        """
        Get all genes in a specific pathway
        
        Args:
            pathway_id: KEGG pathway ID (e.g., "path:hsa05200")
        
        Returns:
            Set of gene IDs in the pathway
        """
        url = f"{self.base_url}/link/{self.organism_code}/{pathway_id}"
        response = self._rate_limited_request(url)
        
        if not response:
            return set()
        
        genes = set()
        for line in response.strip().split('\n'):
            if line:
                parts = line.split('\t')
                if len(parts) >= 2:
                    gene_id = parts[1]  # e.g., "hsa:10327"
                    # Extract just the gene number
                    genes.add(gene_id)
        
        return genes
    
    def get_gene_symbols(self, gene_ids: List[str]) -> Dict[str, str]:
        """
        Convert KEGG gene IDs to gene symbols
        
        Args:
            gene_ids: List of KEGG gene IDs
        
        Returns:
            {gene_id: gene_symbol}
        """
        if not gene_ids:
            return {}
        
        print(f"Fetching gene symbols for {len(gene_ids)} genes...")
        symbols = {}
        
        # Process in batches (KEGG API limits batch size)
        batch_size = 5
        for i in range(0, len(gene_ids), batch_size):
            batch = gene_ids[i:i+batch_size]
            batch_str = '+'.join(batch)
            
            url = f"{self.base_url}/get/{batch_str}"
            response = self._rate_limited_request(url)
            
            if not response:
                continue
            
            # Parse response to extract gene symbols
            current_gene = None
            for line in response.split('\n'):
                if line.startswith('ENTRY'):
                    # e.g., "ENTRY       hsa:10327            CDS       T01001"
                    parts = line.split()
                    if len(parts) >= 2:
                        current_gene = parts[1]
                elif line.startswith('NAME') and current_gene:
                    # e.g., "NAME        CDK11A, CDK11"
                    # Take the first symbol
                    gene_name = line.replace('NAME', '').strip()
                    symbol = gene_name.split(',')[0].strip()
                    symbols[current_gene] = symbol
        
        return symbols
    
    def download_pathways_and_genes(self) -> pd.DataFrame:
        """
        Download all pathways and their associated genes
        
        Returns:
            DataFrame with columns: gene_name, pathway_name
        """
        pathways = self.get_all_pathways()
        
        if not pathways:
            print("ERROR: No pathways found")
            return pd.DataFrame()
        
        print(f"\nFetching genes for each pathway...")
        
        # Collect all gene-pathway pairs
        gene_pathway_pairs = []
        all_genes = set()
        
        for idx, (pathway_id, pathway_name) in enumerate(pathways.items()):
            if idx % 10 == 0:
                print(f"  Processing pathway {idx+1}/{len(pathways)}...")
            
            genes = self.get_genes_in_pathway(pathway_id)
            
            for gene_id in genes:
                gene_pathway_pairs.append({
                    'gene_kegg_id': gene_id,
                    'pathway_name': pathway_name,
                    'pathway_id': pathway_id
                })
                all_genes.add(gene_id)
        
        print(f"\nFound {len(gene_pathway_pairs)} gene-pathway associations")
        print(f"Unique genes: {len(all_genes)}")
        
        # Convert KEGG gene IDs to symbols
        print(f"\nConverting KEGG IDs to gene symbols...")
        gene_symbols = self.get_gene_symbols(list(all_genes))
        
        # Create final dataframe
        final_pairs = []
        for pair in gene_pathway_pairs:
            gene_symbol = gene_symbols.get(pair['gene_kegg_id'], pair['gene_kegg_id'])
            final_pairs.append({
                'gene_name': gene_symbol,
                'pathway_name': pair['pathway_name']
            })
        
        df = pd.DataFrame(final_pairs)
        
        # Remove duplicates (same gene in same pathway from multiple sources)
        df = df.drop_duplicates()
        
        print(f"\nFinal dataset: {len(df)} unique gene-pathway pairs")
        print(f"Unique pathways: {df['pathway_name'].nunique()}")
        
        return df


In [ ]:
organism='human'
kegg_code=None
output='kegg_all_pathways.csv'
rate_limit=0.4
filter_pathways=None
    
# Determine organism code
if kegg_code:
    organism_code = kegg_code
else:
    organism_code = KEGG_ORGANISM_CODES[organism]

print("="*80)
print("KEGG Database Downloader")
print("="*80)
print(f"Organism: {organism} ({organism_code})")
print(f"Output: {output}")
print(f"Rate limit delay: {rate_limit} seconds")
if filter_pathways:
    print(f"Pathway filter: '{filter_pathways}'")
print("="*80)
print("\nNOTE: KEGG API is for academic use only.")
print("Ensure you have permission to access KEGG data for your institution.\n")
    

In [ ]:
try:
    # Download data
    downloader = KEGGDownloader(organism_code, rate_limit)
    df = downloader.download_pathways_and_genes()
    
    if df.empty:
        print("ERROR: No data downloaded")
    
    # Filter pathways if requested
    if filter_pathways:
        original_len = len(df)
        df = df[df['pathway_name'].str.contains(filter_pathways, case=False, na=False)]
        print(f"\nFiltered: {original_len} -> {len(df)} rows")
    
    # Save to CSV
    df.to_csv(output, index=False)
    print(f"\nSaved to: {output}")
    
    # Print summary statistics
    print("\n" + "="*80)
    print("Summary Statistics")
    print("="*80)
    print(f"Total gene-pathway associations: {len(df)}")
    print(f"Unique genes: {df['gene_name'].nunique()}")
    print(f"Unique pathways: {df['pathway_name'].nunique()}")
    
    print("\nTop 10 pathways by gene count:")
    top_pathways = df['pathway_name'].value_counts().head(10)
    for pathway, count in top_pathways.items():
        print(f"  {pathway}: {count} genes")
    
    print("\nExample rows:")
    print(df.head(10).to_string(index=False))
    
    print("\n" + "="*80)
    print("Success!")
    print("="*80)
    

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
organism='human'
kegg_code=None
output='kegg_signaling_pathways.csv'
rate_limit=0.4
filter_pathways="signaling"
    
# Determine organism code
if kegg_code:
    organism_code = kegg_code
else:
    organism_code = KEGG_ORGANISM_CODES[organism]

print("="*80)
print("KEGG Database Downloader")
print("="*80)
print(f"Organism: {organism} ({organism_code})")
print(f"Output: {output}")
print(f"Rate limit delay: {rate_limit} seconds")
if filter_pathways:
    print(f"Pathway filter: '{filter_pathways}'")
print("="*80)
print("\nNOTE: KEGG API is for academic use only.")
print("Ensure you have permission to access KEGG data for your institution.\n")
    

In [ ]:
try:
    # Download data
    downloader = KEGGDownloader(organism_code, rate_limit)
    df = downloader.download_pathways_and_genes()
    
    if df.empty:
        print("ERROR: No data downloaded")
    
    # Filter pathways if requested
    if filter_pathways:
        original_len = len(df)
        df = df[df['pathway_name'].str.contains(filter_pathways, case=False, na=False)]
        print(f"\nFiltered: {original_len} -> {len(df)} rows")
    
    # Save to CSV
    df.to_csv(output, index=False)
    print(f"\nSaved to: {output}")
    
    # Print summary statistics
    print("\n" + "="*80)
    print("Summary Statistics")
    print("="*80)
    print(f"Total gene-pathway associations: {len(df)}")
    print(f"Unique genes: {df['gene_name'].nunique()}")
    print(f"Unique pathways: {df['pathway_name'].nunique()}")
    
    print("\nTop 10 pathways by gene count:")
    top_pathways = df['pathway_name'].value_counts().head(10)
    for pathway, count in top_pathways.items():
        print(f"  {pathway}: {count} genes")
    
    print("\nExample rows:")
    print(df.head(10).to_string(index=False))
    
    print("\n" + "="*80)
    print("Success!")
    print("="*80)
    

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
class FastKEGGDownloader:
    """Download pathway and gene data - fast version without symbol lookup"""
    
    def __init__(self, organism_code: str = 'hsa', rate_limit_delay: float = 0.4):
        self.organism_code = organism_code
        self.rate_limit_delay = rate_limit_delay
        self.base_url = 'https://rest.kegg.jp'
        self.last_request_time = 0
    
    def _rate_limited_request(self, url: str) -> str:
        """Make a rate-limited request to KEGG API"""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.rate_limit_delay:
            time.sleep(self.rate_limit_delay - elapsed)
        
        self.last_request_time = time.time()
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"ERROR fetching {url}: {str(e)}")
            return None
    
    def get_all_pathways(self) -> Dict[str, str]:
        """Get all pathways for the organism"""
        print(f"Fetching pathway list for {self.organism_code}...")
        
        url = f"{self.base_url}/list/pathway/{self.organism_code}"
        response = self._rate_limited_request(url)
        
        if not response:
            return {}
        
        pathways = {}
        for line in response.strip().split('\n'):
            if line:
                parts = line.split('\t')
                if len(parts) >= 2:
                    pathway_id = parts[0]
                    pathway_name = parts[1]
                    pathways[pathway_id] = pathway_name
        
        print(f"Found {len(pathways)} pathways")
        return pathways
    
    def get_genes_in_pathway(self, pathway_id: str) -> List[tuple]:
        """
        Get all genes in a specific pathway with their details
        
        Returns: list of (gene_id, pathway_name) tuples
        """
        url = f"{self.base_url}/link/{self.organism_code}/{pathway_id}"
        response = self._rate_limited_request(url)
        
        if not response:
            return []
        
        genes = []
        for line in response.strip().split('\n'):
            if line:
                parts = line.split('\t')
                if len(parts) >= 2:
                    gene_id = parts[1]  # e.g., "hsa:10327"
                    genes.append(gene_id)
        
        return genes
    
    def download_pathways_and_genes_fast(self) -> pd.DataFrame:
        """
        Download all pathways and genes WITHOUT symbol conversion
        Much faster and avoids network timeouts
        """
        pathways = self.get_all_pathways()
        
        if not pathways:
            print("ERROR: No pathways found")
            return pd.DataFrame()
        
        print(f"\nFetching genes for each pathway...")
        
        gene_pathway_pairs = []
        genes_by_pathway = {}
        
        for idx, (pathway_id, pathway_name) in enumerate(pathways.items()):
            if idx % 50 == 0:
                print(f"  {idx+1}/{len(pathways)} pathways ({100*idx//len(pathways)}%)")
            
            genes = self.get_genes_in_pathway(pathway_id)
            genes_by_pathway[pathway_name] = len(genes)
            
            for gene_id in genes:
                gene_pathway_pairs.append({
                    'gene_name': gene_id,  # Keep as KEGG ID (e.g., hsa:12345)
                    'pathway_name': pathway_name
                })
        
        print(f"  100/{len(pathways)} pathways - Complete!")
        
        print(f"\nFound {len(gene_pathway_pairs)} gene-pathway associations")
        
        # Create dataframe
        df = pd.DataFrame(gene_pathway_pairs)
        
        # Remove duplicates
        df = df.drop_duplicates()
        
        unique_genes = df['gene_name'].nunique()
        unique_pathways = df['pathway_name'].nunique()
        
        print(f"Unique genes: {unique_genes}")
        print(f"Unique pathways: {unique_pathways}")
        
        return df



In [ ]:
organism='human'
kegg_code=None
output="kegg_pathways_fast.csv"
filter_pathways=None

if kegg_code:
    organism_code = kegg_code
else:
    organism_code = KEGG_ORGANISM_CODES[organism]

print("="*80)
print("Fast KEGG Downloader (no symbol conversion)")
print("="*80)
print(f"Organism: {organism} ({organism_code})")
print(f"Output: {output}")
print("="*80)
print()

In [ ]:
try:
    downloader = FastKEGGDownloader(organism_code)
    df = downloader.download_pathways_and_genes_fast()
    
    if df.empty:
        print("ERROR: No data downloaded")
    
    # Filter if requested
    if filter_pathways:
        original_len = len(df)
        df = df[df['pathway_name'].str.contains(filter_pathways, case=False, na=False)]
        print(f"\nFiltered: {original_len} -> {len(df)} rows (searching for '{filter_pathways}')")
    
    # Save
    df.to_csv(output, index=False)
    print(f"\nSaved to: {output}")
    
    # Summary
    print("\n" + "="*80)
    print("Summary")
    print("="*80)
    print(f"Total associations: {len(df)}")
    print(f"Unique genes: {df['gene_name'].nunique()}")
    print(f"Unique pathways: {df['pathway_name'].nunique()}")
    
    print("\nTop 10 pathways by gene count:")
    top_pathways = df['pathway_name'].value_counts().head(10)
    for pathway, count in top_pathways.items():
        print(f"  {pathway}: {count} genes")
    
    print("\nExample rows (gene IDs are in KEGG format):")
    print(df.head(10).to_string(index=False))
    
    print("\n" + "="*80)
    print("SUCCESS!")
    print("="*80)
    print("\nNOTE: Gene names are in KEGG format (e.g., hsa:12345)")
    print("      This is fine - they work perfectly with pathway_distance.py")
    print("      For human-readable names, you can convert them separately")
    print("      or use external gene ID mapping tools.")
    

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()


In [ ]:
organism='human'
kegg_code=None
output='kegg_signaling_pathways_fast.csv'
rate_limit=0.4
filter_pathways="signaling"
    
# Determine organism code
if kegg_code:
    organism_code = kegg_code
else:
    organism_code = KEGG_ORGANISM_CODES[organism]

print("="*80)
print("KEGG Database Downloader")
print("="*80)
print(f"Organism: {organism} ({organism_code})")
print(f"Output: {output}")
print(f"Rate limit delay: {rate_limit} seconds")
if filter_pathways:
    print(f"Pathway filter: '{filter_pathways}'")
print("="*80)
print("\nNOTE: KEGG API is for academic use only.")
print("Ensure you have permission to access KEGG data for your institution.\n")


In [ ]:
try:
    downloader = FastKEGGDownloader(organism_code)
    df = downloader.download_pathways_and_genes_fast()
    
    if df.empty:
        print("ERROR: No data downloaded")
    
    # Filter if requested
    if filter_pathways:
        original_len = len(df)
        df = df[df['pathway_name'].str.contains(filter_pathways, case=False, na=False)]
        print(f"\nFiltered: {original_len} -> {len(df)} rows (searching for '{filter_pathways}')")
    
    # Save
    df.to_csv(output, index=False)
    print(f"\nSaved to: {output}")
    
    # Summary
    print("\n" + "="*80)
    print("Summary")
    print("="*80)
    print(f"Total associations: {len(df)}")
    print(f"Unique genes: {df['gene_name'].nunique()}")
    print(f"Unique pathways: {df['pathway_name'].nunique()}")
    
    print("\nTop 10 pathways by gene count:")
    top_pathways = df['pathway_name'].value_counts().head(10)
    for pathway, count in top_pathways.items():
        print(f"  {pathway}: {count} genes")
    
    print("\nExample rows (gene IDs are in KEGG format):")
    print(df.head(10).to_string(index=False))
    
    print("\n" + "="*80)
    print("SUCCESS!")
    print("="*80)
    print("\nNOTE: Gene names are in KEGG format (e.g., hsa:12345)")
    print("      This is fine - they work perfectly with pathway_distance.py")
    print("      For human-readable names, you can convert them separately")
    print("      or use external gene ID mapping tools.")
    

except Exception as e:
    print(f"\nERROR: {str(e)}")
    import traceback
    traceback.print_exc()


# converting csv kegg ids to uniprot

In [ ]:


import requests
import time
from typing import Dict, List, Union
import pandas as pd

def convert_kegg_to_uniprot(gene_ids: List[str], 
                            rate_limit_delay: float = 0.4,
                            batch_size: int = 50,
                            verbose: bool = True) -> Dict[str, str]:
    """
    Convert KEGG gene IDs to UniProt IDs using KEGG REST conv API
    
    Args:
        gene_ids: List of KEGG gene IDs (e.g., ['hsa:10327', 'hsa:672', 'hsa:675'])
        rate_limit_delay: Delay between API calls in seconds (respects KEGG's 3 calls/sec limit)
        batch_size: Number of genes to convert per API call
        verbose: Print progress information
    
    Returns:
        Dictionary mapping {kegg_id: uniprot_id}
        Unconvertible genes are omitted from the dictionary (use .get() with fallback)
    
    Examples:
        # Get mapping dict
        mapping = convert_kegg_to_uniprot(['hsa:10327', 'hsa:672'])
        print(mapping)  # {'hsa:10327': 'P38398', 'hsa:672': 'P38398'}
        
        # Apply to dataframe
        df['uniprot_id'] = df['kegg_id'].map(mapping).fillna(df['kegg_id'])
    """
    
    if not gene_ids:
        if verbose:
            print("No gene IDs provided")
        return {}
    
    # Remove duplicates but preserve order
    unique_ids = list(dict.fromkeys(gene_ids))
    
    if verbose:
        print(f"Converting {len(unique_ids)} unique KEGG gene IDs to UniProt IDs")
        print(f"(Using KEGG REST conv API - batch size: {batch_size})\n")
    
    uniprot_mapping = {}
    failed_genes = []
    processed = 0
    base_url = 'https://rest.kegg.jp'
    last_request_time = 0
    
    # Process in batches
    for batch_idx in range(0, len(unique_ids), batch_size):
        batch = unique_ids[batch_idx:batch_idx + batch_size]
        batch_str = '+'.join(batch)
        
        # Rate limiting
        elapsed = time.time() - last_request_time
        if elapsed < rate_limit_delay:
            time.sleep(rate_limit_delay - elapsed)
        
        last_request_time = time.time()
        
        # Make API request
        url = f"{base_url}/conv/uniprot/{batch_str}"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            
            # Parse response: each line is "kegg_id\tuniprot_id"
            for line in response.text.strip().split('\n'):
                if line:
                    parts = line.split('\t')
                    if len(parts) >= 2:
                        kegg_id = parts[0]
                        uniprot_id = parts[1]
                        # Remove "up:" prefix if present
                        if uniprot_id.startswith('up:'):
                            uniprot_id = uniprot_id[3:]
                        uniprot_mapping[kegg_id] = uniprot_id
                        processed += 1
            
            # Progress update
            if verbose and (batch_idx + batch_size) % 250 == 0 or (batch_idx + batch_size) >= len(unique_ids):
                pct = 100 * (batch_idx + batch_size) // len(unique_ids)
                print(f"  {min(batch_idx + batch_size, len(unique_ids))}/{len(unique_ids)} "
                      f"({pct}%) - Got {len(uniprot_mapping)} UniProt IDs")
        
        except requests.exceptions.Timeout:
            if verbose:
                print(f"  Batch {batch_idx//batch_size} timed out - will use KEGG IDs as fallback")
            failed_genes.extend(batch)
        
        except requests.exceptions.RequestException as e:
            if verbose:
                print(f"  Error fetching batch: {str(e)}")
            failed_genes.extend(batch)
    
    # Summary
    if verbose:
        print(f"\n{'='*60}")
        print(f"Conversion Results")
        print(f"{'='*60}")
        print(f"Total input genes: {len(unique_ids)}")
        print(f"Successful conversions: {len(uniprot_mapping)}")
        print(f"Failed conversions: {len(failed_genes)}")
        if len(unique_ids) > 0:
            print(f"Success rate: {100*len(uniprot_mapping)//len(unique_ids)}%")
        
        if failed_genes:
            print(f"\nWarning: Could not convert {len(failed_genes)} genes")
            print(f"  (network timeout or unavailable genes)")
            print(f"  Use fallback: df['uniprot_id'] = df['kegg_id'].map(mapping).fillna(df['kegg_id'])")
    
    return uniprot_mapping


def convert_kegg_to_uniprot_mapping(gene_ids: List[str],
                                   rate_limit_delay: float = 0.4,
                                   batch_size: int = 50,
                                   verbose: bool = True) -> Dict[str, str]:
    """
    Alias for convert_kegg_to_uniprot - same function, different name for clarity
    """
    return convert_kegg_to_uniprot(gene_ids, rate_limit_delay, batch_size, verbose)


def convert_single_uniprot_to_symbol(uniprot_id: str, verbose: bool = False) -> str:
    """
    Quick function to convert a single UniProt ID to gene symbol
    
    Args:
        uniprot_id: Single UniProt ID (e.g., 'P38398')
        verbose: Print info
    
    Returns:
        Gene symbol (e.g., 'BRCA1') or original UniProt ID if not found
    
    Examples:
        symbol = convert_single_uniprot_to_symbol('P38398')
        print(symbol)  # Output: BRCA1
        
        # In a dataframe apply
        df['gene_symbol'] = df['uniprot_id'].apply(convert_single_uniprot_to_symbol)
    """
    try:
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            # Extract gene symbol from genes array
            if 'genes' in data and len(data['genes']) > 0:
                # Iterate through genes array
                for gene_entry in data['genes']:
                    # Check for geneName with value
                    if 'geneName' in gene_entry:
                        gene_name_obj = gene_entry['geneName']
                        if isinstance(gene_name_obj, dict) and 'value' in gene_name_obj:
                            symbol = gene_name_obj['value']
                            if symbol:
                                if verbose:
                                    print(f"{uniprot_id} -> {symbol}")
                                return symbol
        
        if verbose:
            print(f"Could not find symbol for {uniprot_id}")
        return uniprot_id
    
    except Exception as e:
        if verbose:
            print(f"Error converting {uniprot_id}: {str(e)}")
        return uniprot_id


def convert_uniprot_to_gene_names(uniprot_ids: List[str],
                                  rate_limit_delay: float = 0.5,
                                  verbose: bool = True) -> Dict[str, str]:
    """
    Convert UniProt IDs to gene names using UniProt REST API
    
    Args:
        uniprot_ids: List of UniProt IDs (e.g., ['P38398', 'P04637'])
        rate_limit_delay: Delay between API calls in seconds
        verbose: Print progress information
    
    Returns:
        Dictionary mapping {uniprot_id: gene_name}
        IDs without gene names are omitted from the dictionary
    
    Examples:
        # Get mapping dict
        mapping = convert_uniprot_to_gene_names(['P38398', 'P04637'])
        print(mapping)  # {'P38398': 'BRCA1', 'P04637': 'TP53'}
        
        # Apply to dataframe
        df['gene_symbol'] = df['uniprot_id'].map(mapping)
    """
    
    if not uniprot_ids:
        if verbose:
            print("No UniProt IDs provided")
        return {}
    
    # Remove duplicates
    unique_ids = list(dict.fromkeys(uniprot_ids))
    
    if verbose:
        print(f"Fetching gene names for {len(unique_ids)} UniProt IDs")
        print(f"(Using UniProt REST API)\n")
    
    gene_mapping = {}
    failed_ids = []
    processed = 0
    last_request_time = 0
    
    # Process one by one (UniProt API doesn't support batch for this query type)
    for idx, uniprot_id in enumerate(unique_ids):
        # Rate limiting
        elapsed = time.time() - last_request_time
        if elapsed < rate_limit_delay:
            time.sleep(rate_limit_delay - elapsed)
        
        last_request_time = time.time()
        
        try:
            # Query UniProt API in JSON format
            url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
            response = requests.get(url, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                
                # Try to extract gene name
                gene_name = None
                
                # Extract gene symbol from genes array
                if 'genes' in data and len(data['genes']) > 0:
                    # Iterate through genes array to find first geneName
                    for gene_entry in data['genes']:
                        if 'geneName' in gene_entry:
                            gene_name_obj = gene_entry['geneName']
                            if isinstance(gene_name_obj, dict) and 'value' in gene_name_obj:
                                gene_name = gene_name_obj['value']
                                break
                
                if gene_name:
                    gene_mapping[uniprot_id] = gene_name
                    processed += 1
                else:
                    failed_ids.append(uniprot_id)
            else:
                failed_ids.append(uniprot_id)
            
            # Progress update
            if verbose and (idx + 1) % 100 == 0 or (idx + 1) == len(unique_ids):
                pct = 100 * (idx + 1) // len(unique_ids)
                print(f"  {idx + 1}/{len(unique_ids)} ({pct}%) - Got {len(gene_mapping)} gene names")
        
        except requests.exceptions.Timeout:
            if verbose and idx == 0:
                print(f"  Note: Some requests may timeout (network dependent)")
            failed_ids.append(uniprot_id)
        
        except requests.exceptions.RequestException as e:
            if verbose and idx < 5:  # Only print first few errors
                print(f"  Error fetching {uniprot_id}: {str(e)}")
            failed_ids.append(uniprot_id)
    
    # Summary
    if verbose:
        print(f"\n{'='*60}")
        print(f"UniProt to Gene Name Conversion Results")
        print(f"{'='*60}")
        print(f"Total input IDs: {len(unique_ids)}")
        print(f"Successful conversions: {len(gene_mapping)}")
        print(f"Failed conversions: {len(failed_ids)}")
        if len(unique_ids) > 0:
            print(f"Success rate: {100*len(gene_mapping)//len(unique_ids)}%")
        
        if failed_ids and len(failed_ids) < 10:
            print(f"\nFailed IDs: {failed_ids}")
    
    return gene_mapping


def apply_full_conversion_pipeline(csv_file: str,
                                   output_file: str = None,
                                   gene_column: str = 'gene_name',
                                   kegg_rate_limit: float = 0.4,
                                   uniprot_rate_limit: float = 0.5,
                                   keep_kegg_fallback: bool = True,
                                   verbose: bool = True) -> pd.DataFrame:
    """
    Complete pipeline: KEGG ID -> UniProt ID -> Gene Name
    
    Loads a CSV with KEGG gene IDs, converts to UniProt, then to gene names.
    Optionally saves output CSV with all three identifier types.
    
    Args:
        csv_file: Path to input CSV file (e.g., 'kegg_pathways_fast.csv')
        output_file: Path to save output CSV (if None, only returns dataframe)
        gene_column: Name of column containing KEGG gene IDs
        kegg_rate_limit: Delay between KEGG API calls (seconds)
        uniprot_rate_limit: Delay between UniProt API calls (seconds)
        keep_kegg_fallback: If True, keep KEGG ID if conversion fails
        verbose: Print progress information
    
    Returns:
        DataFrame with columns: 
            - original gene_column (KEGG IDs)
            - {gene_column}_uniprot (UniProt IDs)
            - {gene_column}_symbol (Gene names)
            - [other original columns like pathway_name]
    
    Examples:
        # Full conversion pipeline
        df = apply_full_conversion_pipeline(
            'kegg_pathways_fast.csv',
            output_file='kegg_pathways_full.csv'
        )
        print(df.head())
        
        # Just signaling pathways
        df = apply_full_conversion_pipeline(
            'kegg_signaling_pathways.csv',
            output_file='signaling_with_names.csv'
        )
    """
    
    if verbose:
        print("="*60)
        print("KEGG ID -> UniProt ID -> Gene Name Conversion Pipeline")
        print("="*60)
        print()
    
    # Step 1: Load CSV
    if verbose:
        print(f"Step 1: Loading {csv_file}...")
    
    df = pd.read_csv(csv_file)
    
    if gene_column not in df.columns:
        raise ValueError(f"Column '{gene_column}' not found in CSV. Available columns: {df.columns.tolist()}")
    
    # Get unique gene IDs
    unique_genes = df[gene_column].unique().tolist()
    
    if verbose:
        print(f"Found {len(df)} rows with {len(unique_genes)} unique genes\n")
    
    # Step 2: Convert KEGG to UniProt
    if verbose:
        print("Step 2: Converting KEGG IDs to UniProt IDs...")
    
    kegg_to_uniprot = convert_kegg_to_uniprot(
        unique_genes, 
        rate_limit_delay=kegg_rate_limit,
        verbose=verbose
    )
    
    # Apply mapping
    if keep_kegg_fallback:
        df[f'{gene_column}_uniprot'] = df[gene_column].map(kegg_to_uniprot).fillna(df[gene_column])
    else:
        df[f'{gene_column}_uniprot'] = df[gene_column].map(kegg_to_uniprot)
        df = df.dropna(subset=[f'{gene_column}_uniprot'])
    
    # Step 3: Convert UniProt to Gene Names
    if verbose:
        print("\nStep 3: Converting UniProt IDs to gene names...")
    
    unique_uniprot = df[f'{gene_column}_uniprot'].unique().tolist()
    # Filter out KEGG IDs (if fallback was used)
    unique_uniprot = [uid for uid in unique_uniprot if not uid.startswith('hsa:')]
    
    uniprot_to_gene = convert_uniprot_to_gene_names(
        unique_uniprot,
        rate_limit_delay=uniprot_rate_limit,
        verbose=verbose
    )
    
    # Apply mapping
    df[f'{gene_column}_symbol'] = df[f'{gene_column}_uniprot'].map(uniprot_to_gene)
    
    # Summary statistics
    if verbose:
        print(f"\n{'='*60}")
        print("Pipeline Complete - Final Results")
        print(f"{'='*60}")
        print(f"Total rows: {len(df)}")
        print(f"Unique genes with symbols: {df[f'{gene_column}_symbol'].notna().sum()}")
        print(f"Unique genes without symbols: {df[f'{gene_column}_symbol'].isna().sum()}")
        print()
        print("Example rows:")
        example_cols = [gene_column, f'{gene_column}_uniprot', f'{gene_column}_symbol']
        if 'pathway_name' in df.columns:
            example_cols.append('pathway_name')
        print(df[example_cols].drop_duplicates(gene_column).head(10).to_string(index=False))
    
    # Optionally save
    if output_file:
        df.to_csv(output_file, index=False)
        if verbose:
            print(f"\nSaved to: {output_file}")
    
    return df


def apply_uniprot_conversion_to_csv(csv_file: str,
                                   output_file: str = None,
                                   gene_column: str = 'gene_name',
                                   rate_limit_delay: float = 0.4,
                                   keep_kegg_fallback: bool = True,
                                   verbose: bool = True) -> pd.DataFrame:
    """
    Load a CSV file (like kegg_pathways_fast.csv) and convert KEGG IDs to UniProt
    
    Args:
        csv_file: Path to input CSV file
        output_file: Path to save output CSV (if None, only returns dataframe)
        gene_column: Name of column containing KEGG gene IDs (default: 'gene_name')
        rate_limit_delay: Delay between API calls in seconds
        keep_kegg_fallback: If True, keep KEGG ID if UniProt conversion fails
        verbose: Print progress information
    
    Returns:
        DataFrame with converted gene IDs
    
    Examples:
        # Convert and save
        df = apply_uniprot_conversion_to_csv(
            'kegg_pathways_fast.csv',
            output_file='kegg_pathways_uniprot.csv'
        )
        
        # Just convert in memory
        df = apply_uniprot_conversion_to_csv('kegg_signaling_pathways.csv')
        print(df.head())
    """
    
    if verbose:
        print(f"Loading {csv_file}...")
    
    df = pd.read_csv(csv_file)
    
    if gene_column not in df.columns:
        raise ValueError(f"Column '{gene_column}' not found in CSV. Available columns: {df.columns.tolist()}")
    
    # Get unique gene IDs
    unique_genes = df[gene_column].unique().tolist()
    
    if verbose:
        print(f"Found {len(df)} rows with {len(unique_genes)} unique genes\n")
    
    # Convert
    mapping = convert_kegg_to_uniprot(unique_genes, rate_limit_delay, verbose=verbose)
    
    # Apply mapping
    if keep_kegg_fallback:
        # Keep KEGG ID if no UniProt ID available
        df[f'{gene_column}_uniprot'] = df[gene_column].map(mapping).fillna(df[gene_column])
    else:
        # Only keep genes that were successfully converted
        df[f'{gene_column}_uniprot'] = df[gene_column].map(mapping)
        df = df.dropna(subset=[f'{gene_column}_uniprot'])
    
    # Optionally save
    if output_file:
        df.to_csv(output_file, index=False)
        if verbose:
            print(f"\nSaved to: {output_file}")
    
    return df



In [ ]:
convert_single_uniprot_to_symbol(df['gene_name_uniprot'][6])


In [ ]:
df_full = apply_full_conversion_pipeline(
    'kegg_pathways_fast.csv',
    output_file='kegg_pathways_with_names.csv'
)
print(df_full.head())
Convert all UniProt IDs in a column
df = pd.read_csv('kegg_pathways_with_names.csv')
# df['gene_symbol'] = df['gene_name_uniprot'].apply(convert_single_uniprot_to_symbol)
# print(df)
uniprot_ids = df['gene_name_uniprot'].unique().tolist()
import time
start = time.time()
gene_names = convert_uniprot_to_gene_names(uniprot_ids)
df['gene_symbol'] = df['gene_name_uniprot'].map(gene_names)
end = time.time()
print(end - start)
df.head()


In [ ]:
df.to_csv("kegg_pathways_with_names_symbols.csv")

In [ ]:
df_full = apply_full_conversion_pipeline(
    'kegg_signaling_pathways_fast.csv',
    output_file='kegg_signaling_fast_pathways_with_names.csv'
)
print(df_full.head())
# df = pd.read_csv('kegg_signaling_fast_pathways_with_names.csv')
# uniprot_ids = df['gene_name_uniprot'].unique().tolist()
# start = time.time()
# gene_names = convert_uniprot_to_gene_names(uniprot_ids)
# df['gene_symbol'] = df['gene_name_uniprot'].map(gene_names)
# end = time.time()
# print(end - start)
# df.head()

In [ ]:
df = pd.read_csv('kegg_signaling_fast_pathways_with_names.csv')
uniprot_ids = df['gene_name_uniprot'].unique().tolist()
start = time.time()
gene_names = convert_uniprot_to_gene_names(uniprot_ids)
df['gene_symbol'] = df['gene_name_uniprot'].map(gene_names)
end = time.time()
print(end - start)
df.head()

In [ ]:
df.to_csv("kegg_signaling_fast_pathways_with_names_symbols.csv")

In [ ]:
df = pd.read_csv('kegg_signaling_pathways_with_names.csv')
uniprot_ids = df['gene_name_uniprot'].unique().tolist()
start = time.time()
gene_names = convert_uniprot_to_gene_names(uniprot_ids)
df['gene_symbol'] = df['gene_name_uniprot'].map(gene_names)
end = time.time()
print(end - start)
df.head()

In [ ]:
df.to_csv("kegg_signaling_pathways_with_names_symbols.csv")

In [ ]:
import requests
import json

# Get all pathways for a specific species (e.g., human)
response = requests.get('https://reactome.org/ContentService/data/species/Homo%20sapiens/pathways')
pathways = response.json()

# Get pathway details including participants
pathway_id = pathways[0]['stId']  # e.g., 'R-HSA-109606'
pathway_details = requests.get(f'https://reactome.org/ContentService/data/pathway/{pathway_id}/containedEvents').json()